In [ ]:
import pandas as pd
import numpy as np

# 1. Danh sách 26 đặc trưng 
SELECTED_COLS = [
    # Nhóm Packet Length
    'Max Packet Length', 'Min Packet Length', 'Average Packet Size', 
    'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Std', 
    'Packet Length Std', 'Total Length of Fwd Packets', 'Subflow Fwd Bytes',
    # Nhóm Time/IAT
    'Flow Duration', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 
    'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Max',
    # Nhóm Flags/Settings
    'ACK Flag Count', 'SYN Flag Count', 'Init_Win_bytes_forward', 
    'min_seg_size_forward', 'Fwd Header Length', 
    # Nhóm Protocol/Rate/Traffic
    'Protocol', 'Fwd Packets/s', 'Bwd Packets/s', 'Total Backward Packets',
    'Label' # Cột nhãn để gán nhãn
]


input_file = '../data/csv-03-11/testing/org/UDPLag.csv'   
output_file = '../data/csv-03-11/testing/cleaned/cleaned_UDPLag.csv' 
chunk_size = 400000 

print(f"Bắt đầu xử lý file: {input_file}")

# Khởi tạo reader đọc theo chunk
reader = pd.read_csv(input_file, chunksize=chunk_size, low_memory=False)

is_first_chunk = True
total_rows = 0

# 3. Tiến hành xử lý
for i, chunk in enumerate(reader):
    # Làm sạch khoảng trắng tên cột
    chunk.columns = chunk.columns.str.strip()
    
    # Lọc 26 cột (bỏ qua nếu file thiếu cột nào đó)
    available_cols = [c for c in SELECTED_COLS if c in chunk.columns]
    df_processed = chunk[available_cols].copy()
    
    # Xử lý Lỗi (Inf, NaN)
    df_processed.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_processed.dropna(inplace=True)
    
    # Gán nhãn nhị phân
    if 'Label' in df_processed.columns:
        df_processed['Label'] = df_processed['Label'].apply(
            lambda x: 0 if str(x).strip().upper() == 'BENIGN' else 1
        )
    
    # Ghi xuống ổ cứng
    if is_first_chunk:
        df_processed.to_csv(output_file, index=False, mode='w')
        is_first_chunk = False
    else:
        df_processed.to_csv(output_file, index=False, mode='a', header=False)
        
    total_rows += len(df_processed)
    print(f" -> Đã xử lý xong chunk {i+1}...")

print(f"Hoàn tất! Đã lưu thành công {total_rows} dòng sạch vào file '{output_file}'.")


Bắt đầu xử lý file: ../data/csv-03-11/testing/org/UDPLag.csv
 -> Đã xử lý xong chunk 1...
 -> Đã xử lý xong chunk 2...
Hoàn tất! Đã lưu thành công 725165 dòng sạch vào file '../data/csv-03-11/testing/cleaned/cleaned_UDPLag.csv'.


In [41]:
import pandas as pd

# Đường dẫn file đã làm sạch (hoặc file tạm)
file_path = '../data/csv-01-12/01-12/cleaned_UDPLag.csv'

benign_count = 0
ddos_count = 0

print("Đang đếm nhãn theo từng chunk...")
for chunk in pd.read_csv(file_path, chunksize=500000):
    counts = chunk['Label'].value_counts()
    benign_count += counts.get(0, 0)
    ddos_count += counts.get(1, 0)

print(f"Kết quả thực tế trong file:")
print(f"- Số dòng Lành tính (0): {benign_count}")
print(f"- Số dòng Tấn công (1): {ddos_count}")
print(f"- Tổng cộng: {benign_count + ddos_count}")

Đang đếm nhãn theo từng chunk...
Kết quả thực tế trong file:
- Số dòng Lành tính (0): 3705
- Số dòng Tấn công (1): 366900
- Tổng cộng: 370605


In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

input_file = '../data/CSV-03-11/testing/cleaned/cleaned_LDAP.csv'
output_file = '../data/csv-03-11/testing/scaled/LSTM_LDAP_scaled.csv'
chunk_size = 500000 

print( "ĐANG QUÉT TÌM VỊ TRÍ NHÃN 0 (BENIGN)")
benign_indices = []
total_rows = 0

# Đọc lần 1: Chỉ lấy index của nhãn 0 và đếm tổng số dòng
for chunk in pd.read_csv(input_file, chunksize=chunk_size, low_memory=False):
    # chunk.index tự động khớp với vị trí dòng thực tế trong file gốc
    idx_0 = chunk[chunk['Label'] == 0].index.tolist()
    benign_indices.extend(idx_0)
    total_rows += len(chunk)

n_benign = len(benign_indices)
print(f"Hoàn tất Pass 1. Tổng số dòng trong file: {total_rows}")
print(f"Tổng số nhãn 0 thu thập được: {n_benign}")

# TÍNH TOÁN CÁC INDEX CẦN GIỮ LẠI ĐỂ ĐẠT TỶ LỆ 44 - 56 
# Nếu n_benign chiếm 44%, thì tổng số dòng cần có là: n_benign / 0.44
target_total_rows = int(n_benign / 0.37)
target_ddos_rows = target_total_rows - n_benign

print(f"Số lượng nhãn 1 (DDoS) cần lấy xung quanh nhãn 0: {target_ddos_rows}")

# Khởi tạo tập hợp chứa các index cần lấy (bắt đầu bằng việc chứa toàn bộ nhãn 0)
indices_to_keep = set(benign_indices)

# Thuật toán Mở rộng cửa sổ động (Dynamic Window Expansion)
window = 1
while len(indices_to_keep) < target_total_rows and window < total_rows:
    for idx in benign_indices:
        # Mở rộng về phía trước 1 dòng
        if idx - window >= 0:
            indices_to_keep.add(idx - window)
        
        # Mở rộng về phía sau 1 dòng
        if idx + window < total_rows:
            indices_to_keep.add(idx + window)
            
        # Dừng ngay nếu đã gom đủ số lượng mong muốn
        if len(indices_to_keep) >= target_total_rows:
            break
            
    window += 1 # Tăng độ rộng cửa sổ lên nếu vẫn chưa đủ dữ liệu

print(f"Đã tính toán xong các khối liên tục. Kích thước cửa sổ neo tối đa đã lan rộng: {window} dòng mỗi bên.")

# Chuyển set thành list và sắp xếp lại để đảm bảo trật tự thời gian tuyệt đối
final_indices_to_extract = sorted(list(indices_to_keep))

# --- PASS 2: RÚT TRÍCH DỮ LIỆU ---
print(" RÚT TRÍCH DỮ LIỆU TỪ FILE")
sampled_chunks = []

for i, chunk in enumerate(pd.read_csv(input_file, chunksize=chunk_size, low_memory=False)):
    # Tìm xem trong chunk hiện tại có những index nào nằm trong danh sách cần bốc
    # Biến đổi final_indices_to_extract thành dạng có thể check nhanh, nhưng intersection của index là tối ưu nhất
    valid_indices = chunk.index.intersection(final_indices_to_extract)
    
    if len(valid_indices) > 0:
        sampled_chunks.append(chunk.loc[valid_indices])
    print(f"Đã rút trích xong chunk {i+1}...")

# Gộp tất cả các mảnh đã rút trích lại (Dữ liệu đã tự động chuẩn thời gian vì final_indices_to_extract đã được sort)
df_sampled = pd.concat(sampled_chunks)

# --- BƯỚC CUỐI: CHUẨN HÓA DỮ LIỆU (Min-Max Scaling) ---
print(" BẮT ĐẦU CHUẨN HÓA DỮ LIỆU")
# Tách Label
y = df_sampled['Label'].reset_index(drop=True)
X = df_sampled.drop(columns=['Label'])

scaler = MinMaxScaler()
X_scaled_array = scaler.fit_transform(X)

# Chuyển về DataFrame và ghép Label lại
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)
df_final = pd.concat([X_scaled, y], axis=1)

# Lưu file
df_final.to_csv(output_file, index=False)

print("-" * 40)
print(f"HOÀN TẤT! Đã lưu file tại: {output_file}")
print(f"Kích thước tập dữ liệu cuối cùng: {df_final.shape[0]} dòng, {df_final.shape[1]} cột.")
print(f"Tỷ lệ nhãn thực tế sau khi lấy mẫu:\n{df_final['Label'].value_counts(normalize=True) * 100}")

ĐANG QUÉT TÌM VỊ TRÍ NHÃN 0 (BENIGN)
Hoàn tất Pass 1. Tổng số dòng trong file: 2113234
Tổng số nhãn 0 thu thập được: 5124
Số lượng nhãn 1 (DDoS) cần lấy xung quanh nhãn 0: 8724
Đã tính toán xong các khối liên tục. Kích thước cửa sổ neo tối đa đã lan rộng: 4 dòng mỗi bên.
 RÚT TRÍCH DỮ LIỆU TỪ FILE
Đã rút trích xong chunk 1...
Đã rút trích xong chunk 2...
Đã rút trích xong chunk 3...
Đã rút trích xong chunk 4...
Đã rút trích xong chunk 5...
 BẮT ĐẦU CHUẨN HÓA DỮ LIỆU
----------------------------------------
HOÀN TẤT! Đã lưu file tại: ../data/csv-03-11/testing/scaled/LSTM_LDAP_scaled.csv
Kích thước tập dữ liệu cuối cùng: 13849 dòng, 26 cột.
Tỷ lệ nhãn thực tế sau khi lấy mẫu:
Label
1    63.000939
0    36.999061
Name: proportion, dtype: float64
